In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: xpu


In [2]:
class TicTacToe:
    # n=3 for normal tic tac toe, could probably generalize this more but whatever
    def __init__(self, n=3):
        self.n = n
        self.board = np.zeros(n*n)
        self.current_player = 1

    def reset(self):
        self.board = np.zeros(self.n*self.n)
        self.current_player = 1
        return self.board.copy()

    def available_actions(self):
        # indices where board is still 0
        avail = []
        for i in range(len(self.board)):
            if self.board[i] == 0:
                avail.append(i)
        return avail

    def step(self, action):
        if self.board[action] != 0:
            print("tried an invalid move??", action)
            return self.board.copy(), -10, True

        self.board[action] = self.current_player

        if self.check_win(self.current_player):
            return self.board.copy(), 10, True

        if len(self.available_actions()) == 0:
            # board full, no winner = draw
            return self.board.copy(), 0, True

        self.current_player *= -1
        return self.board.copy(), 0, False

    def check_win(self, p):
        b = self.board.reshape(self.n, self.n)

        for i in range(self.n):
            if np.all(b[i, :] == p):
                return True
            if np.all(b[:, i] == p):
                return True

        # diagonals
        if np.all(np.diag(b) == p):
            return True
        if np.all(np.diag(np.fliplr(b)) == p):
            return True

        return False

    def render(self):
        # quick debug helper, not really used in training
        symbols = {1: 'X', -1: 'O', 0: '.'}
        b = self.board.reshape(self.n, self.n)
        for row in b:
            print(' '.join(symbols[int(x)] for x in row))


In [ ]:
from collections import namedtuple

Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward', 'done'))

class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

memory = ReplayMemory(5000)


In [4]:
class DQN(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.layer1 = nn.Linear(size, 256)
        self.layer2 = nn.Linear(256, 256)
        self.layer3 = nn.Linear(256, size)
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.layer3(x)


In [ ]:
board_size = 9
model = DQN(board_size).to(device)
target_model = DQN(board_size).to(device)
target_model.load_state_dict(model.state_dict())
optimizer = optim.Adam(model.parameters(), lr=1e-3)

gamma = 0.99
eps = 1.0
eps_min = 0.05
eps_decay = 0.9995
tau = 0.01  # soft update rate for target network, applied periodically during training


def get_canonical_state(board, player):
    # network always sees the board as "my pieces = 1, opponent = -1", regardless
    # of whether we're actually player 1 or player -1. Without this, the same
    # network has to separately learn how to play as X and as O.
    return board * player


def select_action(canonical_state, available_actions, current_eps):
    if np.random.rand() <= current_eps:
        return random.choice(available_actions)

    state_t = torch.FloatTensor(canonical_state).unsqueeze(0).to(device)
    with torch.no_grad():
        q_values = model(state_t)

    # mask out illegal moves so argmax cant pick them
    mask = torch.full((board_size,), -float('inf'), device=device)
    mask[available_actions] = 0
    return torch.argmax(q_values + mask).item()


def train_step(batch_size):
    global eps

    if len(memory) < batch_size:
        return

    batch = memory.sample(batch_size)

    states = torch.FloatTensor(np.array([t.state for t in batch])).to(device)
    actions = torch.LongTensor([t.action for t in batch]).to(device)
    rewards = torch.FloatTensor([t.reward for t in batch]).to(device)
    next_states = torch.FloatTensor(np.array([t.next_state for t in batch])).to(device)
    dones = torch.FloatTensor([float(t.done) for t in batch]).to(device)

    # Q(s, a) for the actions actually taken -- one batched forward pass
    q_values = model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # next_state is stored canonically from the OPPONENT's perspective (it's their
    # turn to move next). Tic-tac-toe is zero-sum, so their best move is bad for
    # us -> subtract their max Q instead of adding it. Use target_model here for
    # stable bootstrapping targets.
    with torch.no_grad():
        next_q = target_model(next_states).max(1)[0]
        targets = rewards - gamma * next_q * (1.0 - dones)

    loss = nn.SmoothL1Loss()(q_values, targets)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if eps > eps_min:
        eps *= eps_decay


In [ ]:
N = 3
env = TicTacToe(n=N)
episodes = 5000  # tuned to finish in ~1-2 min; ~2000 episodes already hits ~97% winrate vs random
batch_size = 64
target_update_every = 20  # episodes, soft-updates target_model throughout training

for e in range(episodes):
    state = env.reset()

    for t in range(N*N):
        player = env.current_player
        canon_state = get_canonical_state(state, player)
        actions = env.available_actions()
        action = select_action(canon_state, actions, eps)

        next_state, reward, done = env.step(action)
        # next mover is env.current_player after step (unchanged if the game ended)
        canon_next_state = get_canonical_state(next_state, env.current_player)
        memory.push(canon_state, action, canon_next_state, reward, done)

        train_step(batch_size)  # train on every move, not just once per episode

        state = next_state
        if done:
            break

    if (e + 1) % target_update_every == 0:
        for tp, p in zip(target_model.parameters(), model.parameters()):
            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)

    if (e+1) % 500 == 0:
        print(f"Episode: {e+1}/{episodes}, Epsilon: {eps:.2f}")

print(f"done training, N={N}")


In [ ]:
def evaluate(env, num_games=100):
    w = 0
    d = 0
    l = 0

    for g in range(num_games):
        state = env.reset()
        done = False

        while not done:
            player = env.current_player
            canon_state = get_canonical_state(state, player)
            actions = env.available_actions()
            action = select_action(canon_state, actions, current_eps=0)
            state, reward, done = env.step(action)

            if done:
                if reward == 10:
                    w += 1
                elif reward == 0:
                    d += 1
                break

            # opponent just plays randomly for now
            actions = env.available_actions()
            action = random.choice(actions)
            state, reward, done = env.step(action)

            if done:
                if reward == 10:
                    l += 1
                elif reward == 0:
                    d += 1
                break

    print(f'over {num_games} games vs random:')
    print(f'wins: {w} ({w/num_games*100:.1f}%)')
    print(f'draws: {d} ({d/num_games*100:.1f}%)')
    print(f'losses: {l} ({l/num_games*100:.1f}%)')

evaluate(env)


In [ ]:
torch.save(model.state_dict(), "best_model.pth")

In [ ]:
import os
os.listdir()

In [ ]:
from google.colab import files
files.download("best_model.pth")